In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = "/content/drive/MyDrive/FPX_Project"
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f"Drive mounted. Project root: {DRIVE_ROOT}")

Mounted at /content/drive
Drive mounted. Project root: /content/drive/MyDrive/FPX_Project


In [2]:
import os

DRIVE_ROOT = "/content/drive/MyDrive/FPX_Project"
folders = [
    "config", "utils", "data",
    "checkpoints/fl_sim", "checkpoints/signals",
    "checkpoints/interface", "checkpoints/eval",
    "outputs", "logs"
]
for f in folders:
    os.makedirs(os.path.join(DRIVE_ROOT, f), exist_ok=True)

print("Folder structure created:")
for f in folders:
    print(f"  {DRIVE_ROOT}/{f}")

Folder structure created:
  /content/drive/MyDrive/FPX_Project/config
  /content/drive/MyDrive/FPX_Project/utils
  /content/drive/MyDrive/FPX_Project/data
  /content/drive/MyDrive/FPX_Project/checkpoints/fl_sim
  /content/drive/MyDrive/FPX_Project/checkpoints/signals
  /content/drive/MyDrive/FPX_Project/checkpoints/interface
  /content/drive/MyDrive/FPX_Project/checkpoints/eval
  /content/drive/MyDrive/FPX_Project/outputs
  /content/drive/MyDrive/FPX_Project/logs


In [3]:
checkpoint_utils_code = '''
import os, json, pickle, datetime

DRIVE_ROOT = "/content/drive/MyDrive/FPX_Project"

def get_path(*parts):
    path = os.path.join(DRIVE_ROOT, *parts)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    return path

def log(message):
    ts = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{ts}] {message}"
    print(line)
    log_path = get_path("logs", "project_log.txt")
    with open(log_path, "a") as f:
        f.write(line + "\\n")

def save_json(data, subdir, name):
    path = get_path(subdir, f"{name}.json")
    with open(path, "w") as f:
        json.dump(data, f, indent=2, default=str)
    log(f"Saved JSON checkpoint: {subdir}/{name}.json")
    return path

def load_json(subdir, name, default=None):
    path = get_path(subdir, f"{name}.json")
    if not os.path.exists(path):
        return default
    with open(path) as f:
        return json.load(f)

def save_pickle(obj, subdir, name):
    path = get_path(subdir, f"{name}.pkl")
    with open(path, "wb") as f:
        pickle.dump(obj, f)
    log(f"Saved pickle checkpoint: {subdir}/{name}.pkl")
    return path

def load_pickle(subdir, name, default=None):
    path = get_path(subdir, f"{name}.pkl")
    if not os.path.exists(path):
        return default
    with open(path, "rb") as f:
        return pickle.load(f)

def list_checkpoints(subdir):
    d = os.path.join(DRIVE_ROOT, subdir)
    if not os.path.exists(d):
        return []
    return sorted(os.listdir(d))

def project_status():
    print("=== FPX Project Drive Status ===")
    for subdir in ["config", "data", "checkpoints/fl_sim", "checkpoints/signals",
                   "checkpoints/interface", "checkpoints/eval", "outputs", "logs"]:
        files = list_checkpoints(subdir)
        print(f"{subdir}: {len(files)} file(s)")
        for f in files[:10]:
            print(f"   - {f}")
'''

util_path = os.path.join(DRIVE_ROOT, "utils", "checkpoint_utils.py")
with open(util_path, "w") as f:
    f.write(checkpoint_utils_code)

print(f"checkpoint_utils.py written to: {util_path}")

checkpoint_utils.py written to: /content/drive/MyDrive/FPX_Project/utils/checkpoint_utils.py


In [4]:
import sys
sys.path.insert(0, os.path.join(DRIVE_ROOT, "utils"))
import checkpoint_utils as cu

# Initial config — dataset choice is a placeholder for now (Open Question in taxonomy doc),
# starting with sklearn breast cancer (zero download friction, fast iteration).
# Swap to a richer biomedical/tabular set later by changing one field.
project_config = {
    "project_name": "Federation Process Explainability (FPX)",
    "dataset": "breast_cancer_sklearn",   # placeholder, swappable
    "num_clients": 8,
    "non_iid_alpha": 0.3,                  # Dirichlet concentration (lower = more skewed)
    "num_rounds": 20,
    "model_type": "small_mlp",
    "created": str(__import__("datetime").datetime.now()),
    "colab_gpu": "T4",
    "current_phase": "P2_setup"
}

cu.save_json(project_config, "config", "project_config")
print("Config saved:")
print(project_config)

[2026-09-02 09:53:52] Saved JSON checkpoint: config/project_config.json
Config saved:
{'project_name': 'Federation Process Explainability (FPX)', 'dataset': 'breast_cancer_sklearn', 'num_clients': 8, 'non_iid_alpha': 0.3, 'num_rounds': 20, 'model_type': 'small_mlp', 'created': '2026-09-02 09:53:52.025905', 'colab_gpu': 'T4', 'current_phase': 'P2_setup'}


In [6]:
cu.project_status()

=== FPX Project Drive Status ===
config: 1 file(s)
   - project_config.json
data: 0 file(s)
checkpoints/fl_sim: 0 file(s)
checkpoints/signals: 0 file(s)
checkpoints/interface: 0 file(s)
checkpoints/eval: 0 file(s)
outputs: 0 file(s)
logs: 1 file(s)
   - project_log.txt
